# Learning `RegistryBase` through a small object collection

Sooner or later, one `Nematics3D` object needs to keep track of several other named objects: several figures, several analysis planes, several defect lines, or some other collection.

A plain Python list can preserve order, and a dictionary can look things up by name. `RegistryBase` combines the behaviors that `Nematics3D` repeatedly needs: **ordered storage, lookup by name or index, unique names, and a relation from each registered object back to the registry that currently contains it.**

This tutorial focuses only on the small set of operations needed to use a registry confidently.


## Make a few objects to organize

We will create three simple `SmoothedLine` objects. Their geometry is not important here; we only need ordinary named `Nematics3D` objects that can be placed in a registry.


In [ ]:
import numpy as np
import nematics3d as n3d


def make_line(name, phase=0.0):
    x = np.linspace(0.0, 2.0 * np.pi, 61)
    coords = np.column_stack([x, np.sin(x + phase), np.zeros_like(x)])
    return n3d.SmoothedLine(
        coords,
        name=name,
        window_length=7,
        min_line_length=2,
    )


line_a = make_line("line-a")
line_b = make_line("line-b", phase=0.5)


## Create a registry and put objects into it

A registry starts empty. Register objects with `act_register()`. The method returns the object that was registered.


In [ ]:
registry = n3d.RegistryBase("example lines")
registry.act_register(line_a)
registry.act_register(line_b)

registry


## Retrieve an object by the name you gave it

The most useful registry operation is usually name lookup:


In [ ]:
registry["line-a"] is line_a


Because registration order is preserved, integer indexing also works:


In [ ]:
registry[0] is line_a, registry[1] is line_b


A registry also behaves naturally when you need to inspect the whole collection.


In [ ]:
print(len(registry))
print([obj.name for obj in registry])


At this point, the basic mental model is already enough for most use:

```text
named objects
    ↓ act_register()
RegistryBase
    ├── lookup by name
    ├── lookup by insertion-order index
    └── iterate in insertion order
```


## The registry keeps names unique

Names are used as lookup keys, so two objects in the same registry cannot keep the same name. If you register another object named `line-a`, `RegistryBase` resolves the collision automatically by adding a suffix.


In [ ]:
another = registry.act_register(make_line("line-a", phase=1.0))

print(another.name)
print([obj.name for obj in registry])


This matters because later code can safely use the object's current `name` as a registry key without first checking whether that name was already occupied.


## A registered object knows which registry contains it

For normal `ClassBase` objects, registration also establishes a `registry` relation back to the collection.


In [ ]:
line_a.registry is registry


So the relationship is available in both directions:

```text
registry["line-a"]  ─────→  line_a
        ↑                       │
        └──── line_a.registry ──┘
```

This is one reason `RegistryBase` is more useful inside `Nematics3D` than a bare dictionary. The collection is part of the object model rather than only a container sitting beside it.


## Registering the same object elsewhere moves it

An object normally belongs to one registry at a time. Registering it in another registry moves it from the old collection to the new one and updates its `registry` relation.


In [ ]:
other = n3d.RegistryBase("other lines")
other.act_register(line_a)

print(line_a.registry is other)
print(line_a in other)
print(line_a in registry)


## Remove objects when they no longer belong to the collection

Use `act_unregister()` for one object or `act_clear()` for the whole registry.


In [ ]:
other.act_unregister(line_a)
print(line_a.registry)
print(len(other))


## If you only need the current contents

`registry.entity` gives the registered objects as a tuple. It is useful when another piece of code needs a read-only snapshot of the current collection rather than access to the registry operations themselves.


In [ ]:
registry.entity


## You do not need to memorize more than this

For ordinary use, the essential interface is small:

```python
registry.act_register(obj)
registry["name"]
registry[index]
len(registry)
for obj in registry:
    ...
registry.act_unregister(obj)
registry.act_clear()
```

`RegistryBase` inherits the same inspection vocabulary used elsewhere in `Nematics3D`, so `registry.show_` plus autocomplete remains available when you need to discover more.


## The main idea

Use `RegistryBase` when several named `Nematics3D` objects belong to one managed collection. It gives you stable insertion order, lookup by name or index, automatic name uniqueness, and a synchronized `registry` relation on registered objects.

The key conceptual difference from a plain list or dictionary is that registration is **part of the object relationship**, not merely storage.
